# Task adapters scored against human labels, both forms of scaling

**GPU: NVIDIA A100.** Runtime > Change runtime type > A100. Run `colab_pilot.ipynb` first: it says which tasks to keep and prints the expected run time of each stage on your GPU.

Each LoRA adapter is trained on the training split of one task (PubMedQA, Yelp star rating, GSM8K, SAMSum) and scored on fixed test sets with human labels as a function of the scaling factor, in logit space and in weight space.

| Stage | Content |
|---|---|
| A | Qwen2.5-7B-Instruct, seed 42: four adapters (with checkpoints at 10%, 20%, 40% of the steps); own task on the full grid; KL to the base model; the other three tasks and the general benchmarks on the reduced grid |
| B | seeds 43 and 44: four adapters each; own task on the full grid; the two multiple-choice tasks as cross tasks |
| C | Phi-4-mini-instruct, seed 42: stage A without the checkpoints |
| D | the checkpoints of stage A: own task at factors 0.5 to 3 |

Set `STAGES` in the configuration cell to run a subset. **Everything is resumable**: after a disconnection, run all cells again; finished adapters, factors and files are skipped. Results go to `MyDrive/dynalora_results/level4/`.

## 0. Setup

In [ ]:
import os, subprocess, sys
GPU = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print('GPU:', GPU or 'none')
assert GPU, 'No GPU: Runtime > Change runtime type'
EXPECTED = ('A100',)
ALLOW_OTHER_GPU = False   # set to True to run on another GPU anyway (it is recorded in run_info.json)
if not any(g in GPU for g in EXPECTED) and not ALLOW_OTHER_GPU:
    raise SystemExit(f'This notebook is planned for {EXPECTED} (bfloat16, 40 GB or more), got {GPU}: Runtime > Change runtime type')

REPO = '/content/dynalora'
BRANCH = 'paper'
if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} https://github.com/alexroehrs/dynalora.git {REPO}
%cd {REPO}
!git fetch -q origin {BRANCH} && git checkout -q {BRANCH} && git reset -q --hard origin/{BRANCH}
!git log -1 --format='commit %H  %s'
!pip install -q -r requirements.txt
# Colab ships torchao 0.10, and peft 0.21 then refuses to load any LoRA adapter (it requires torchao > 0.16).
# Nothing in this repository uses torchao, so it is removed.
!pip uninstall -y -q torchao

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RESULTS = '/content/drive/MyDrive/dynalora_results'   # results root (Google Drive); results are not kept in the repository
L4 = f'{RESULTS}/level4'
os.makedirs(L4, exist_ok=True)
sys.path.insert(0, REPO)
from dynalora.runinfo import record_session
e = record_session(f'{L4}/run_info.json', 'run notebook start', seeds={'adapter_training': 'per adapter (s<seed> in its name)', 'train_subsets': 42, 'test_subsets': 0, 'bootstrap': 0})
print(e['gpu'], e.get('gpu_memory_gb'), e['git'], e['libraries'], sep='\n')
import torch
assert torch.cuda.is_bf16_supported(), 'this GPU has no bfloat16 support'
GRAD_CKPT = '--gradient_checkpointing' if torch.cuda.get_device_properties(0).total_memory < 60 * 2 ** 30 else ''   # 40 GB: checkpointing in training
print('gradient checkpointing in training:', bool(GRAD_CKPT))
FAILED = []   # steps that did not finish; reported at the end

CPU tests (random tiny models, about 2 min).

In [ ]:
!python -m pytest -q tests/test_tasks.py

## Configuration

In [ ]:
def sh(cmd, log):
    """Runs a shell command, appends its output to `log`, returns the exit code."""
    print(cmd)
    get_ipython().system(f"bash -c 'set -o pipefail; {cmd} 2>&1 | tee -a {log}'")
    return get_ipython().user_ns.get('_exit_code', 1)

def step(name, cmd, log, required=False):
    """Runs one resumable step (the scripts skip what is already saved). A failure stops the notebook if
    `required`, else it is recorded in FAILED and the notebook goes on."""
    rc = sh(cmd, log)
    if rc != 0:
        msg = f'{name} (exit code {rc}, see {log})'
        if required:
            raise RuntimeError(f'{msg}: rerun the notebook to resume')
        FAILED.append(msg); print(f'WARNING: {msg}; going on, rerun the notebook to resume')
    return rc == 0

def tag(model):
    return model.rstrip('/').split('/')[-1]

def eval_bs(model, task):
    """Batch size of the scoring: generation tasks decode short contexts, multiple-choice tasks read long ones."""
    small = 'mini' in model.lower()
    gen = task in ('gsm8k', 'samsum')
    return (32 if gen else 16) if small else (16 if gen else 8)

STAGES = ['A', 'B', 'C', 'D']            # remove a letter to skip a stage
PRIMARY = 'Qwen/Qwen2.5-7B-Instruct'
SECOND = 'microsoft/Phi-4-mini-instruct'
TASKS = ['pubmedqa', 'yelp5', 'gsm8k', 'samsum']   # edit if the pilot rejected a task
SEED0, MORE_SEEDS = 42, [43, 44]
REDUCED = '0 0.5 1 1.5 2'                # reduced grid of the cross-task and benchmark scores
TRUNC = '0.5 1 1.25 1.5 2 3'             # factors of the truncated-training stage
KL_ITEMS, BENCH_LIMIT = 200, 500
A_DIR, S_DIR, B_DIR = f'{L4}/adapters', f'{L4}/scores', f'{L4}/bench'
for d in (A_DIR, S_DIR, B_DIR):
    os.makedirs(d, exist_ok=True)

def name(model, task, seed, ckpt=None):
    return f'{tag(model)}__{task}__s{seed}' + (f'__ckpt{ckpt}' if ckpt else '')

def train(model, task, seed, ckpts=False):
    return step(f'train {name(model, task, seed)}',
                f'python scripts/train_task_chat.py --model {model} --task {task} --seed {seed} --adapters_dir {A_DIR} {GRAD_CKPT}'
                + (' --ckpt_fracs 0.1 0.2 0.4' if ckpts else ''), f'{L4}/train.log')

def own(model, task, seed, ckpt=None, grid=None, kl=True):
    n = name(model, task, seed, ckpt)
    return step(f'own {n}', f'python scripts/eval_task.py --model {model} --task {task} --adapter_dir {A_DIR}/{n} --out_dir {S_DIR}/{n} '
                f'--base_cache {S_DIR}/_base --batch_size {eval_bs(model, task)}'
                + (f' --kl_items {KL_ITEMS}' if kl else '') + (f' --lambdas {grid} --lambdas_weight {grid}' if grid else ''),
                f'{L4}/score.log')

def cross(model, task, seed, other):
    n = name(model, task, seed)
    return step(f'cross {n} on {other}', f'python scripts/eval_task.py --model {model} --task {other} --adapter_task {task} '
                f'--adapter_dir {A_DIR}/{n} --out_dir {S_DIR}/{n} --base_cache {S_DIR}/_base --batch_size {eval_bs(model, other)} '
                f'--lambdas {REDUCED} --lambdas_weight {REDUCED}', f'{L4}/score.log')

def bench(model, task, seed):
    return step(f'bench {name(model, task, seed)}', f'python scripts/zeroshot_bench.py --model {model} --adapters_dir {A_DIR} '
                f'--configs {task}__s{seed} --dtype bfloat16 --batch_size {16 if "mini" in model.lower() else 4} --limit {BENCH_LIMIT} '
                f'--lambdas {REDUCED} --out {B_DIR}', f'{L4}/bench.log')

def core(model, ckpts):
    """Stage A or C: train, own task, then cross tasks, then benchmarks (the cheapest complete result first)."""
    trained = [t for t in TASKS if train(model, t, SEED0, ckpts)]
    for t in trained:
        own(model, t, SEED0)
    for t in trained:
        for o in TASKS:
            if o != t:
                cross(model, t, SEED0, o)
    for t in trained:
        bench(model, t, SEED0)

In [ ]:
# The four datasets load from the Hub and have the expected fields (a few seconds each; stops here otherwise).
from dynalora import tasks as T
for task in TASKS:
    tr, te = T.load_examples(task, 'train', n=300), T.load_examples(task, 'test', n=20)
    print(f'{task}: {len(tr)} train, {len(te)} test | input: {te[0]["input"][:70]!r} | answer: {te[0]["answer"][:50]!r}')

## Stage A. Qwen2.5-7B-Instruct, seed 42

In [ ]:
if 'A' in STAGES:
    core(PRIMARY, ckpts=True)
print('not finished:', FAILED or 'nothing')

## Stage B. Training seeds 43 and 44

In [ ]:
if 'B' in STAGES:
    for seed in MORE_SEEDS:
        for t in TASKS:
            if train(PRIMARY, t, seed):
                own(PRIMARY, t, seed, kl=False)
    for seed in MORE_SEEDS:
        for t in TASKS:
            for o in ('pubmedqa', 'yelp5'):
                if o != t and o in TASKS and os.path.exists(f'{A_DIR}/{name(PRIMARY, t, seed)}/train_log.json'):
                    cross(PRIMARY, t, seed, o)
print('not finished:', FAILED or 'nothing')

## Stage C. Phi-4-mini-instruct, seed 42

In [ ]:
if 'C' in STAGES:
    core(SECOND, ckpts=False)
print('not finished:', FAILED or 'nothing')

## Stage D. Adapters saved at 10%, 20% and 40% of the training steps

In [ ]:
if 'D' in STAGES:
    for t in TASKS:
        for ckpt in (10, 20, 40):
            if os.path.exists(f'{A_DIR}/{name(PRIMARY, t, SEED0, ckpt)}/train_log.json'):
                own(PRIMARY, t, SEED0, ckpt=ckpt, grid=TRUNC, kl=False)
            else:
                print(f'{name(PRIMARY, t, SEED0, ckpt)}: checkpoint not found (stage A trains it)')
print('not finished:', FAILED or 'nothing')

## Summary of what is on Drive

In [ ]:
import glob, pandas as pd
rows = []
for d in sorted(glob.glob(f'{S_DIR}/*__s*')):
    for f in sorted(glob.glob(f'{d}/*__*.csv')):
        if os.path.basename(f).startswith(('own', 'cross')):
            df = pd.read_csv(f)
            rows.append(dict(adapter=os.path.basename(d), file=os.path.basename(f), factors=df.factor.nunique(), items=df.item.nunique(),
                             at_0=df[df.factor == 0].score.mean(), at_1=df[df.factor == 1].score.mean(), best=df.groupby('factor').score.mean().max()))
print(pd.DataFrame(rows).round(3).to_string(index=False) if rows else 'no scores yet')
print('benchmark files:', len(glob.glob(f'{B_DIR}/bench__*.csv.gz')))
print('not finished:', FAILED or 'nothing')